# Badger_vision Training Demo

Production-ready training with Badger Factory dataset support.

### Supported Datasets
| Source | Layout |
|--------|--------|
| Badger Factory – Detection/Keypoints | `yolo/images/{train,val}.7z` + `labels/{train,val}.7z` |
| Badger Factory – Classification | `classifier/evolving_ds_*/{train,val}.7z` |
| COCO Export | `{train,val}.7z` with `coco_instances.json` |
| Plain YOLO / COCO | Already extracted |

Archives: `.zip` `.7z` `.tar.gz` `.tar.bz2` `.tar.xz` `.rar`

## 1 — Install & GPU Check

In [ ]:
import subprocess
import sys

for pkg in ["tqdm", "py7zr", "rarfile"]:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

import torch

print(f"PyTorch : {torch.__version__}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"GPU     : {gpu_name}  ({gpu_mem:.1f} GB)")
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
    DEVICE = torch.device("cuda:0")
else:
    print("WARNING: No GPU detected — training will be slow.")
    DEVICE = torch.device("cpu")

## 2 — Configure

Point to your dataset and choose the task.

In [ ]:
# ╔════════════════════════════════════════════════════════════╗
# ║                    CONFIGURE HERE                         ║
# ╚════════════════════════════════════════════════════════════╝

DATASET_PATH = "/path/to/your/dataset"  # folder or archive
TASK = "detection"    # "detection" | "keypoints" | "classification"
EPOCHS = 100
BATCH_SIZE = 0        # 0 = auto from GPU VRAM
IMG_SIZE = 640
LR = 0.01
MODEL = "resnext"     # "resnext" or "convnext"
RESUME = None          # path to checkpoint to resume, or None

## 3 — Prepare Dataset

In [ ]:
import importlib.util
import shutil
from pathlib import Path

NOTEBOOK_DIR = Path(".").resolve()
REPO_ROOT = NOTEBOOK_DIR.parent if (NOTEBOOK_DIR.parent / "pyproject.toml").exists() else NOTEBOOK_DIR

spec = importlib.util.spec_from_file_location("linux_train", str(REPO_ROOT / "notebooks" / "linux_train.py"))
lt = importlib.util.module_from_spec(spec)
spec.loader.exec_module(lt)

WORKSPACE = Path("badger_vision_workspace").resolve()
WORKSPACE.mkdir(parents=True, exist_ok=True)

dataset_path = Path(DATASET_PATH).resolve()
assert dataset_path.exists(), f"Dataset not found: {dataset_path}"

if dataset_path.is_file() and lt.is_archive(dataset_path):
    extract_dest = WORKSPACE / "dataset"
    if extract_dest.exists():
        shutil.rmtree(extract_dest)
    dataset_root = lt.extract_archive(dataset_path, extract_dest)
else:
    dataset_root = dataset_path

dataset_root = lt.resolve_dataset_root(dataset_root, TASK)
fmt = lt.detect_format(dataset_root)
print(f"Dataset root : {dataset_root}")
print(f"Format       : {fmt}")

if fmt == "badger_yolo":
    cls = dataset_root / "classes.txt"
    data_info = lt.prepare_badger_yolo(dataset_root, WORKSPACE, cls if cls.exists() else None)
elif fmt == "badger_classifier":
    data_info = lt.prepare_badger_classifier(dataset_root, WORKSPACE)
elif fmt == "coco_archive":
    data_info = lt.prepare_coco_archive(dataset_root, WORKSPACE)
elif fmt == "yolo_flat":
    data_info = lt.prepare_yolo_flat(dataset_root, WORKSPACE)
elif fmt == "coco_flat":
    data_info = lt.prepare_coco_flat(dataset_root, WORKSPACE)
else:
    raise RuntimeError(f"Unknown dataset format in {dataset_root}")

num_classes = lt.detect_num_classes(data_info["train_ann_file"])
num_train = lt.count_images(data_info["train_ann_file"])
print(f"Classes      : {num_classes}")
print(f"Train images : {num_train}")

## 4 — Train

tqdm progress bars per batch + epoch snapshot (progress %, ETA, best loss, GPU mem).

In [ ]:
batch_size = BATCH_SIZE
if batch_size <= 0 and torch.cuda.is_available():
    gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    batch_size = lt.auto_batch_size(gpu_mem_gb, IMG_SIZE)
    print(f"Auto batch size: {batch_size}")
elif batch_size <= 0:
    batch_size = 2

model_cfg, data_cfg = lt.write_configs(
    WORKSPACE, data_info, num_classes,
    epochs=EPOCHS, batch_size=batch_size,
    img_size=IMG_SIZE, lr=LR, model_type=MODEL,
)

lt.run_training(model_cfg, data_cfg, data_info, DEVICE,
                epochs=EPOCHS, batch_size=batch_size, resume=RESUME)

## 5 — Results

In [ ]:
import glob

runs = sorted(glob.glob("runs/train_*"))
if runs:
    latest = runs[-1]
    best_ckpt = f"{latest}/checkpoint_best.pt"
    last_ckpt = f"{latest}/checkpoint_last.pt"
    ckpt_path = best_ckpt if Path(best_ckpt).exists() else last_ckpt
    if Path(ckpt_path).exists():
        ckpt = torch.load(ckpt_path, map_location="cpu")
        print(f"Checkpoint : {ckpt_path}")
        print(f"Epoch      : {ckpt.get('epoch', '?')}")
        print(f"Best Loss  : {ckpt.get('best_loss', '?'):.4f}")
    else:
        print("No checkpoint found.")
else:
    print("No training runs found.")